<table style="width: 100%; border-collapse: collapse; border: none; background: #f8fafc; border-left: 6px solid #1e3a8a; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #0f172a; font-size: 2.1em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        Taller Hands-On: Proyecto de ML de Punta a Punta — Entregas Retrasadas 🚚
      </h1>
      <p style="margin: 6px 0 0 0; color: #1e3a8a; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Machine Learning
      </p>
      <p style="margin: 4px 0 0 0; color: #64748b; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #1e3a8a; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        Taller Práctico Evaluativo · Módulo 09
      </span><br>
      <span style="color: #64748b; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #2563eb; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Machine%20Learning/homeworks/09_Casos_Estudio_Aplicaciones_Hands_On.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## Objetivos de Aprendizaje 🔎

Este taller **integra el Módulo 09 (Casos de estudio y aplicaciones)**: recorrerás el **flujo completo de un proyecto de ML** sobre un problema nuevo, una empresa de mensajería que quiere **anticipar las entregas que llegarán tarde** para intervenirlas a tiempo. Cada reto es una etapa del flujo que viste en el cuaderno [00 · Casos de estudio y aplicaciones](../09%20-%20Casos%20de%20Estudio%20y%20Aplicaciones/00_Casos_de_Estudio_y_Aplicaciones.ipynb); el problema es **desbalanceado**, tiene **faltantes** y mezcla variables **numéricas y categóricas**.

Al terminar serás capaz de:

1. **Preparar los datos sin fuga**: partición estratificada y un `ColumnTransformer` que imputa, escala y codifica, ajustado solo con el entrenamiento.
2. **Comparar modelos con una línea base** y una validación cruzada repetida dentro de un `Pipeline`.
3. **Fijar un umbral de decisión por costo del negocio**, no por costumbre (0.5).
4. **Explicar y auditar** el modelo (importancia por permutación, análisis de errores por segmento) y redactar una **ficha del modelo** con sus límites.

> 🔗 Los casos de visión, lenguaje y recomendación del módulo comparten este mismo flujo: [visión](../09%20-%20Casos%20de%20Estudio%20y%20Aplicaciones/01_Vision_por_Computador.ipynb), [PLN](../09%20-%20Casos%20de%20Estudio%20y%20Aplicaciones/02_Procesamiento_de_Lenguaje_Natural.ipynb) y [recomendación](../09%20-%20Casos%20de%20Estudio%20y%20Aplicaciones/03_Sistemas_de_Recomendacion.ipynb).

---
## Instrucciones

- Resuelve cada reto **de forma autónoma**: completa las celdas con `# TODO:` y ejecuta la celda de **autoverificación** que le sigue (los `assert` fallan con un mensaje claro hasta que el reto esté bien resuelto).
- Los retos **se encadenan**: usa las variables que creaste en los anteriores. No modifiques la celda de preparación ni las autoverificaciones.
- Antes de entregar usa *Restart & Run All*: el cuaderno debe correr de principio a fin (< 1 min) sin errores y sin ningún `# TODO` pendiente.
- Todo usa semillas fijas y datos sintéticos: no necesitas descargar nada.

---
### 📦 Preparación del entorno y de los datos

Ejecuta esta celda sin modificarla. Crea `df`: 2 500 entregas de una empresa de mensajería (sintéticas). La variable objetivo es `retraso` (1 = llegó tarde, ≈ 20 %). `peso_kg` y `distancia_km` tienen valores faltantes.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.model_selection import train_test_split, RepeatedStratifiedKFold, cross_val_score, cross_val_predict
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score, confusion_matrix
import warnings; warnings.filterwarnings("ignore")

rng = np.random.default_rng(2026)
n = 2500
df = pd.DataFrame({
    "distancia_km": rng.gamma(3.0, 4.0, n).round(1),
    "peso_kg": rng.gamma(2.0, 3.0, n).round(1),
    "hora_pico": rng.integers(0, 2, n),
    "lluvia_mm": np.where(rng.random(n) < 0.35, rng.exponential(6.0, n), 0.0).round(1),
    "n_paradas": rng.integers(1, 12, n),
    "zona": rng.choice(["norte", "sur", "centro", "rural"], n, p=[0.3, 0.25, 0.3, 0.15]),
    "transportista": rng.choice(["A", "B", "C"], n, p=[0.5, 0.3, 0.2]),
})
eta = (-5.6 + 0.14 * df.distancia_km + 0.10 * df.lluvia_mm + 1.0 * df.hora_pico + 0.18 * df.n_paradas
       + 1.2 * (df.zona == "rural") + 0.6 * (df.transportista == "C"))
df["retraso"] = (rng.random(n) < 1 / (1 + np.exp(-eta))).astype(int)
df.loc[rng.random(n) < 0.08, "peso_kg"] = np.nan
df.loc[rng.random(n) < 0.05, "distancia_km"] = np.nan

X = df.drop(columns="retraso")
y = df["retraso"]
num_cols = ["distancia_km", "peso_kg", "lluvia_mm", "n_paradas"]
cat_cols = ["zona", "transportista"]
bin_cols = ["hora_pico"]
print(df.shape, "| tasa de retraso:", round(y.mean(), 3), "| faltantes:", df.isna().sum()[df.isna().sum() > 0].to_dict())
df.head()

---
## Reto 1: Preparación de Datos sin Fuga 🧹 *(20 puntos)*

**Contexto.** La primera etapa de cualquier proyecto: entender el problema, **sellar la prueba** y construir un preprocesamiento reproducible que aprenda **solo del entrenamiento**.

**Enunciado.**
1. `tasa_retraso`: proporción de retrasos en `y`. Separa `X`, `y` en `X_tr`, `X_te`, `y_tr`, `y_te` (25 % de prueba, `stratify=y`, `random_state=0`).
2. `cols_faltantes`: lista de columnas de `X_tr` con algún valor faltante.
3. Construye `prep`, un `ColumnTransformer` con tres ramas:
   - `"num"` sobre `num_cols`: un `Pipeline` con pasos `"imp"` (`SimpleImputer(strategy="median")`) y `"esc"` (`StandardScaler()`);
   - `"cat"` sobre `cat_cols`: un `Pipeline` con `"imp"` (`SimpleImputer(strategy="most_frequent")`) y `"ohe"` (`OneHotEncoder(handle_unknown="ignore", sparse_output=False)`);
   - `"bin"`: `"passthrough"` sobre `bin_cols`.
4. Ajusta `prep` **solo con `X_tr`** y guarda `Z_tr = prep.transform(X_tr)` y `Z_te = prep.transform(X_te)`.

**Criterios de aceptación.** Partición estratificada (625 filas de prueba); `Z_tr` y `Z_te` sin `NaN`, con **12 columnas** (4 numéricas + 7 *dummies* + 1 binaria); y los parámetros de imputación aprendidos **coinciden con las medianas de `X_tr`** (no de `X`).

In [ ]:
# TODO 1: tasa global y partición estratificada
tasa_retraso = None
X_tr = X_te = y_tr = y_te = None

# TODO 2: columnas del entrenamiento con valores faltantes
cols_faltantes = None

# TODO 3: ColumnTransformer con las ramas "num", "cat" y "bin"
prep = None

# TODO 4: ajustar SOLO con el entrenamiento y transformar ambos conjuntos
Z_tr = None
Z_te = None
print("Tasa de retraso:", tasa_retraso, "| columnas con faltantes:", cols_faltantes)
print("Z_tr:", None if Z_tr is None else Z_tr.shape, "| Z_te:", None if Z_te is None else Z_te.shape)

In [ ]:
# ✅ Autoverificación del Reto 1 (no modifiques esta celda)
assert tasa_retraso is not None and 0.1 < tasa_retraso < 0.4, "Reto 1: tasa_retraso es y.mean() (≈ 0.2)."
assert X_tr is not None and X_tr.shape == (1875, 7) and X_te.shape == (625, 7), "Reto 1: la partición debe ser 75 % / 25 % (1875 y 625 filas)."
assert abs(y_tr.mean() - y_te.mean()) < 0.01, "Reto 1: la partición debe ser estratificada."
assert set(cols_faltantes) == {"peso_kg", "distancia_km"}, f"Reto 1: las columnas con faltantes son peso_kg y distancia_km, no {cols_faltantes}."
assert isinstance(prep, ColumnTransformer), "Reto 1: prep debe ser un ColumnTransformer."
assert {name for name, _, _ in prep.transformers} == {"num", "cat", "bin"}, "Reto 1: las ramas deben llamarse 'num', 'cat' y 'bin'."
assert Z_tr is not None and Z_tr.shape == (1875, 12) and Z_te.shape == (625, 12), f"Reto 1: se esperaban 12 columnas (4 numéricas + 7 dummies + 1 binaria); obtuviste {None if Z_tr is None else Z_tr.shape}."
assert not np.isnan(Z_tr).any() and not np.isnan(Z_te).any(), "Reto 1: tras el preprocesamiento no debe quedar ningún NaN."
_med = prep.named_transformers_["num"].named_steps["imp"].statistics_
assert np.allclose(_med, X_tr[num_cols].median().to_numpy()), "Reto 1: el imputador debe aprender las medianas de X_tr (ajusta prep SOLO con el entrenamiento, no con X)."
print("✅ Reto 1 superado.")

---
## Reto 2: Línea Base y Comparación de Modelos 🧪 *(20 puntos)*

**Contexto.** Antes de elegir, hay que saber **a cuánto equivale no hacer nada** y comparar candidatos con una validación que no dependa de una sola partición.

**Enunciado.** Con el entrenamiento (`X_tr`, `y_tr`):
1. Crea `cv = RepeatedStratifiedKFold(n_splits=3, n_repeats=2, random_state=0)` (6 evaluaciones por modelo).
2. Define el diccionario `modelos` con tres `Pipeline` de dos pasos, `("prep", clone(prep))` y `("clf", ...)`:
   - `"base"`: `DummyClassifier(strategy="prior")`;
   - `"logistica"`: `LogisticRegression(max_iter=1000)`;
   - `"bosque"`: `RandomForestClassifier(n_estimators=60, min_samples_leaf=5, random_state=0, n_jobs=1)`.
3. Calcula con `cross_val_score(..., scoring="roc_auc")` los 6 puntajes de cada uno y guárdalos en `puntajes` (diccionario nombre → arreglo).
4. Construye `tabla`, un `DataFrame` con índice = nombre del modelo y columnas `media` y `desv` (desviación estándar), ordenado de mayor a menor `media`. Guarda en `mejor` el nombre del mejor modelo **que no sea la línea base**.

**Criterios de aceptación.** 6 puntajes por modelo; la línea base ≈ 0.5; el mejor modelo supera a la base por más de 0.15; el preprocesamiento va **dentro** de cada *pipeline* (sin fuga entre pliegues).

In [ ]:
# TODO 1: validación cruzada repetida
cv = None

# TODO 2: tres pipelines (preprocesamiento + clasificador)
modelos = {
    "base": None,
    "logistica": None,
    "bosque": None,
}

# TODO 3: puntajes ROC-AUC de cada modelo (arreglos de 6 valores)
puntajes = {}

# TODO 4: tabla resumen ordenada y mejor modelo distinto de la base
tabla = None
mejor = None
print(tabla)
print("Mejor modelo:", mejor)

In [ ]:
# ✅ Autoverificación del Reto 2 (no modifiques esta celda)
assert cv is not None and cv.get_n_splits() == 6, "Reto 2: cv debe ser RepeatedStratifiedKFold(3 pliegues x 2 repeticiones) = 6 evaluaciones."
assert set(modelos) == {"base", "logistica", "bosque"} and all(isinstance(m, Pipeline) for m in modelos.values()), "Reto 2: modelos debe contener tres Pipeline ('base', 'logistica', 'bosque')."
assert all(m.steps[0][0] == "prep" and isinstance(m.steps[0][1], ColumnTransformer) for m in modelos.values()), "Reto 2: el primer paso de cada Pipeline debe ser ('prep', ColumnTransformer) para que el preprocesamiento se ajuste en cada pliegue."
assert isinstance(modelos["base"].steps[-1][1], DummyClassifier) and isinstance(modelos["bosque"].steps[-1][1], RandomForestClassifier), "Reto 2: revisa los clasificadores de cada pipeline."
assert set(puntajes) == set(modelos) and all(len(v) == 6 for v in puntajes.values()), "Reto 2: puntajes debe tener 6 valores de ROC-AUC por modelo."
assert abs(np.mean(puntajes["base"]) - 0.5) < 0.02, "Reto 2: la línea base 'prior' debe tener AUC ≈ 0.5."
assert tabla is not None and list(tabla.columns) == ["media", "desv"] and set(tabla.index) == set(modelos), "Reto 2: tabla debe tener columnas ['media', 'desv'] e índice con los 3 modelos."
assert tabla["media"].is_monotonic_decreasing, "Reto 2: ordena la tabla de mayor a menor media."
assert mejor in ("logistica", "bosque") and tabla.loc[mejor, "media"] == tabla.drop(index="base")["media"].max(), "Reto 2: mejor es el mejor modelo distinto de 'base'."
assert tabla.loc[mejor, "media"] > tabla.loc["base", "media"] + 0.15, "Reto 2: el mejor modelo debería superar a la línea base por más de 0.15 de AUC."
print("✅ Reto 2 superado.")

---
## Reto 3: Decidir por Costo, No por 0.5 💰 *(20 puntos)*

**Contexto.** Anticipar un retraso permite llamar al cliente o reasignar la ruta. Un retraso **no anticipado** (falso negativo) cuesta **15** unidades (reclamos, reembolso); una intervención **innecesaria** (falso positivo) cuesta **2**. El umbral 0.5 ignora esta asimetría.

**Enunciado.**
1. Implementa `costo(y_real, y_pred)` $= 15\cdot FN + 2\cdot FP$ (con `confusion_matrix` o con máscaras de NumPy).
2. Con el pipeline `modelos[mejor]` obtén probabilidades **fuera de muestra del entrenamiento**: `p_val = cross_val_predict(clone(modelos[mejor]), X_tr, y_tr, cv=5, method="predict_proba")[:, 1]`.
3. Recorre la rejilla `umbrales = np.linspace(0.02, 0.9, 45)` y elige `umbral_opt`, el que **minimiza el costo sobre `p_val`**. Guarda `costo_val_opt` y `costo_val_05` (costo de `p_val` con umbral 0.5).
4. Ajusta `modelo_final = clone(modelos[mejor]).fit(X_tr, y_tr)` y evalúa en la **prueba sellada**: `p_te`, `costo_opt` (con `umbral_opt`), `costo_05` (con 0.5) y `costo_nada` (nunca intervenir: todas las predicciones en 0).

**Criterios de aceptación.** `umbral_opt < 0.5`; `costo_val_opt <= costo_val_05`; en la prueba `costo_opt < costo_05 < costo_nada` (el umbral por costo debe ganar).

In [ ]:
C_FN, C_FP = 15, 2

def costo(y_real, y_pred):
    """Costo total = C_FN * falsos negativos + C_FP * falsos positivos."""
    # TODO 1
    pass

# TODO 2: probabilidades fuera de muestra del entrenamiento con el mejor pipeline
p_val = None

# TODO 3: barrido de umbrales sobre p_val
umbrales = np.linspace(0.02, 0.9, 45)
umbral_opt = None
costo_val_opt = None
costo_val_05 = None

# TODO 4: modelo final y evaluación en la prueba sellada
modelo_final = None
p_te = None
costo_opt = None
costo_05 = None
costo_nada = None
print("Umbral óptimo:", umbral_opt, "| costo validación (opt / 0.5):", costo_val_opt, costo_val_05)
print("Costo en prueba -> óptimo:", costo_opt, "| 0.5:", costo_05, "| no intervenir:", costo_nada)

In [ ]:
# ✅ Autoverificación del Reto 3 (no modifiques esta celda)
assert costo(np.array([1, 1, 0, 0]), np.array([0, 1, 1, 0])) == 15 + 2, "Reto 3: costo([1,1,0,0], [0,1,1,0]) debe ser 15 (un FN) + 2 (un FP) = 17."
assert costo(np.array([1, 0]), np.array([1, 0])) == 0, "Reto 3: sin errores el costo es 0."
assert p_val is not None and p_val.shape == (1875,) and 0 <= p_val.min() and p_val.max() <= 1, "Reto 3: p_val son 1875 probabilidades fuera de muestra (cross_val_predict con method='predict_proba')."
assert umbral_opt is not None and 0 < umbral_opt < 0.5, f"Reto 3: con un FN 7.5 veces más caro que un FP el umbral óptimo debe ser < 0.5 (obtuviste {umbral_opt})."
assert abs(costo_val_opt - costo(y_tr, (p_val >= umbral_opt).astype(int))) < 1e-9, "Reto 3: costo_val_opt debe ser el costo de p_val con umbral_opt."
assert costo_val_opt <= costo_val_05, "Reto 3: el costo con el umbral óptimo no puede superar al del umbral 0.5."
assert modelo_final is not None and hasattr(modelo_final, "predict_proba"), "Reto 3: ajusta modelo_final con el entrenamiento."
assert p_te is not None and p_te.shape == (625,), "Reto 3: p_te son las 625 probabilidades de la prueba."
assert costo_nada == 15 * int(y_te.sum()), "Reto 3: 'no intervenir' cuesta 15 por cada retraso de la prueba."
assert costo_opt < costo_05 < costo_nada, f"Reto 3: en la prueba debe cumplirse costo_opt < costo_05 < costo_nada; obtuviste {costo_opt}, {costo_05}, {costo_nada}."
print("✅ Reto 3 superado.")

---
## Reto 4: Explicar, Auditar y Documentar 🔍📋 *(20 puntos)*

**Contexto.** Antes de desplegar hay que saber **qué mira el modelo**, **dónde falla** y dejar por escrito sus límites (la «ficha del modelo»).

**Enunciado.**
1. **Importancia por permutación** sobre la prueba y el *pipeline completo* (así se barajan las columnas originales): `r = permutation_importance(modelo_final, X_te, y_te, scoring="roc_auc", n_repeats=5, random_state=0, n_jobs=1)`. Construye `imp`, una `Series` con `r.importances_mean` indexada por `X_te.columns` y ordenada de mayor a menor; `top3` es la lista de sus 3 primeros nombres.
2. **Análisis de errores por segmento.** Con `pred = (p_te >= umbral_opt)`, calcula `recall_por_zona`: el *recall* (retrasos detectados / retrasos reales) de la prueba para cada valor de `zona` (una `Series`). `zona_peor` es la zona con menor *recall*.
3. **Ficha del modelo.** Crea el diccionario `ficha` con las claves `"modelo"` (texto), `"auc_prueba"` (AUC en la prueba), `"umbral"` (`umbral_opt`), `"costo_prueba"` (`costo_opt`), `"variables_clave"` (`top3`) y `"limitaciones"` (lista con **al menos 2** textos no vacíos escritos por ti).

**Criterios de aceptación.** `imp` ordenada con la distancia entre las 3 variables más importantes; `recall_por_zona` con las 4 zonas y valores en $[0,1]$; `zona_peor` consistente; `ficha` completa y coherente con tus resultados.

In [ ]:
# TODO 1: importancia por permutación (pipeline completo, prueba sellada)
imp = None
top3 = None

# TODO 2: recall por zona con el umbral de costo
pred = None
recall_por_zona = None
zona_peor = None

# TODO 3: ficha del modelo
ficha = {
    "modelo": None,
    "auc_prueba": None,
    "umbral": None,
    "costo_prueba": None,
    "variables_clave": None,
    "limitaciones": [],
}
print(imp)
print(recall_por_zona, "| peor zona:", zona_peor)
print(ficha)

In [ ]:
# ✅ Autoverificación del Reto 4 (no modifiques esta celda)
assert imp is not None and isinstance(imp, pd.Series) and set(imp.index) == set(X_te.columns), "Reto 4: imp debe ser una Series con una importancia por cada columna original de X_te."
assert imp.is_monotonic_decreasing, "Reto 4: ordena imp de mayor a menor."
assert top3 is not None and top3 == list(imp.index[:3]), "Reto 4: top3 son los nombres de las 3 variables más importantes."
assert "distancia_km" in top3, f"Reto 4: la distancia debería estar entre las 3 variables más importantes (obtuviste {top3}); ¿permutaste con el pipeline completo y scoring='roc_auc'?"
assert pred is not None and pred.shape == (625,), "Reto 4: pred son las 625 decisiones de la prueba con umbral_opt."
assert recall_por_zona is not None and set(recall_por_zona.index) == {"norte", "sur", "centro", "rural"}, "Reto 4: recall_por_zona debe tener una entrada por cada zona."
assert recall_por_zona.between(0, 1).all(), "Reto 4: el recall de cada zona está entre 0 y 1."
_chk = {z: pred[(X_te["zona"] == z).to_numpy() & (y_te.to_numpy() == 1)].mean() for z in recall_por_zona.index}
assert all(abs(recall_por_zona[z] - _chk[z]) < 1e-9 for z in _chk), "Reto 4: recall = retrasos detectados / retrasos reales, calculado solo sobre las filas de cada zona."
assert zona_peor == recall_por_zona.idxmin(), "Reto 4: zona_peor es la zona con menor recall."
assert isinstance(ficha, dict) and {"modelo", "auc_prueba", "umbral", "costo_prueba", "variables_clave", "limitaciones"} <= set(ficha), "Reto 4: la ficha debe tener las seis claves pedidas."
assert abs(ficha["auc_prueba"] - roc_auc_score(y_te, p_te)) < 1e-9 and ficha["umbral"] == umbral_opt and ficha["costo_prueba"] == costo_opt, "Reto 4: la ficha debe reflejar tus resultados (auc_prueba, umbral y costo_prueba)."
assert list(ficha["variables_clave"]) == top3 and isinstance(ficha["modelo"], str) and ficha["modelo"].strip(), "Reto 4: variables_clave = top3 y modelo es un texto."
assert isinstance(ficha["limitaciones"], list) and len([t for t in ficha["limitaciones"] if isinstance(t, str) and t.strip()]) >= 2, "Reto 4: escribe al menos 2 limitaciones (textos no vacíos)."
print("✅ Reto 4 superado.")

---
## Reto 5: Conclusiones 🧾 *(20 puntos)*

Redacta **5 a 8 líneas** como si presentaras el proyecto a la gerencia de operaciones, con tus números:

- ¿Cuánto mejora el mejor modelo a la línea base y qué tan estable es esa mejora entre pliegues? (Reto 2)
- ¿Cuánto ahorra el umbral por costo frente al 0.5 y frente a no intervenir? ¿Qué pasa si el costo de un falso positivo sube? (Reto 3)
- ¿Qué variables dominan y en qué zona falla más el modelo? ¿Qué harías al respecto? (Reto 4)
- ¿Qué **riesgos** ves para desplegarlo (cambio de distribución, datos sintéticos, equidad entre zonas) y qué monitorearías?

---
#### ✍️ Tus conclusiones

_Escribe aquí tus 5 a 8 líneas. Reemplaza este texto por tu análisis._

---
### 🧠 Rúbrica de evaluación

| Criterio | Puntos |
|---|---|
| Reto 1: partición estratificada y `ColumnTransformer` sin fuga | 20 |
| Reto 2: línea base y comparación con CV repetida dentro de `Pipeline` | 20 |
| Reto 3: umbral por costo evaluado en la prueba sellada | 20 |
| Reto 4: importancia, análisis de errores por zona y ficha del modelo | 20 |
| Reto 5: conclusiones numéricas con riesgos de despliegue | 20 |
| **Total** | **100** |

### ✅ Checklist de entrega

- [ ] No queda ningún `# TODO` ni variable en `None`.
- [ ] Las cuatro celdas de autoverificación imprimen «✅ Reto N superado».
- [ ] *Restart & Run All* corre de principio a fin sin errores.
- [ ] La celda de conclusiones está escrita con tus propias palabras y con números.
- [ ] No modificaste la celda de preparación ni las autoverificaciones.

---
<div align="center">
  <p style="font-size: 0.9em; color: #64748b;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Asignatura: Machine Learning</i>
  </p>
</div>